# Getting started with pyAFQ - ParticipantAFQ

In [ ]:
import os
import os.path as op

import matplotlib.pyplot as plt
import nibabel as nib
import plotly

from AFQ.api.participant import ParticipantAFQ
import AFQ.data.fetch as afd
import AFQ.definitions.image as afm

## Example data

The following call downloads a a single subject's data from the Healthy Brain
Network Processed Open Diffusion Derivatives dataset (HBN-POD2) [^1], [^2]
and organizes it in BIDS in the user's home directory under:

`~/AFQ_data/HBN/`

The data is also placed in a derivatives directory, signifying that it has
already undergone the required preprocessing necessary for pyAFQ to run.

In [ ]:
afd.fetch_hbn_preproc(["NDARAA948VFH"])

## Defining data files

If your data is not in BIDS format, you can still use pyAFQ. If you have BIDS
compliant dataset, you can use `GroupAFQ` instead ([plot_001_group_afq_api](plot_001_group_afq_api)).
Otherwise, You will need to define the data files that you want to use. In
this case, we will define the data files for the subject we downloaded above.
The data files are located in the `~/AFQ_data/HBN/derivatives/qsiprep`
directory, and are organized into a BIDS compliant directory structure. The
data files are located in the `dwi` directories.

In [ ]:
sub_dir = op.join(afd.afq_home, "HBN", "derivatives", "qsiprep",
                   "sub-NDARAA948VFH")
dwi_data_file = op.join(sub_dir, "ses-HBNsiteRU", "dwi", (
    "sub-NDARAA948VFH_"
    "ses-HBNsiteRU_"
    "acq-64dir_space-T1w_desc-preproc_dwi.nii.gz"))
bval_file = op.join(sub_dir, "ses-HBNsiteRU", "dwi", (
    "sub-NDARAA948VFH_"
    "ses-HBNsiteRU_"
    "acq-64dir_space-T1w_desc-preproc_dwi.bval"))
bvec_file = op.join(sub_dir, "ses-HBNsiteRU", "dwi", (
    "sub-NDARAA948VFH_"
    "ses-HBNsiteRU_"
    "acq-64dir_space-T1w_desc-preproc_dwi.bvec"))
t1_file = op.join(sub_dir, "anat",
                  "sub-NDARAA948VFH_desc-preproc_T1w.nii.gz")

# You will also need to define the output directory where you want to store the
# results. The output directory needs to exist before exporting ParticipantAFQ
# results.

output_dir = op.join(afd.afq_home, "HBN",
                     "derivatives", "afq", "sub-NDARAA948VFH",
                     "ses-HBNsiteRU", "dwi")
os.makedirs(output_dir, exist_ok=True)

## Set tractography parameters (optional)
We make create a `tracking_params` variable, which we will pass to the
ParticipantAFQ object which specifies that we want 100,000 seeds randomly
distributed in the white matter. We only do this to make this example faster
and consume less space; normally, we use more seeds.

In [ ]:
tracking_params = dict(n_seeds=1e5,
                       random_seeds=True,
                       rng_seed=2025,
                       trx=True)

## Define PVE images (optional)
To improve segmentation and tractography results, we can provide
partial volume estimate (PVE) images for the cerebrospinal fluid (CSF),
gray matter (GM), and white matter (WM). Here, we define these images
using the AFQ.definitions.image.PVEImages class, which takes as input
three AFQ.definitions.image.ImageFile objects, one for each tissue type.
One can also provide a single PVE image with all three tissue types
using the AFQ.definitions.image.PVEImage class. Finally, by default,
if no PVE images are provided, pyAFQ will use SynthSeg2 to compute
these images.

In [ ]:
pve = afm.PVEImages(
    afm.ImageFile(
        path=op.join(sub_dir, "anat", 
                     "sub-NDARAA948VFH_label-CSF_probseg.nii.gz")),
    afm.ImageFile(
        path=op.join(sub_dir, "anat", 
                     "sub-NDARAA948VFH_label-GM_probseg.nii.gz")),
    afm.ImageFile(
        path=op.join(sub_dir, "anat", 
                     "sub-NDARAA948VFH_label-WM_probseg.nii.gz")))

## Brain Mask Definition (optional)

By default, pyAFQ will compute a brain mask from the T1. However,
this requires onnxruntime to be installed. If you do not have onnxruntime
installed, or if you want to use a different brain mask, you can specify
it here.

In [ ]:
brain_mask_definition = afm.ImageFile(
    path=op.join(sub_dir, "anat", "sub-NDARAA948VFH_desc-brain_mask.nii.gz"))

## Initialize a ParticipantAFQ object:

Creates a ParticipantAFQ object, that encapsulates tractometry. This object
can be used to manage the entire [tractometry pipeline](/explanations/index), including:

- Tractography
- Registration
- Segmentation
- Cleaning
- Profiling
- Visualization

To initialize the object, we will pass in the diffusion data files and specify
the output directory where we want to store the results. We will also
pass in the tracking parameters we defined above.

In [ ]:
myafq = ParticipantAFQ(
    dwi_data_file=dwi_data_file,
    bval_file=bval_file,
    bvec_file=bvec_file,
    t1_file=t1_file,
    output_dir=output_dir,
    tracking_params=tracking_params,
    pve=pve,
    brain_mask_definition=brain_mask_definition,
)

## Calculating DTI FA (Diffusion Tensor Imaging Fractional Anisotropy)

The ParticipantAFQ object has a method called `export`, which allows the user
to calculate various derived quantities from the data.

For example, FA can be computed using the DTI model, by explicitly
calling `myafq.export("dti_fa")`. This triggers the computation of DTI
parameters, and stores the results in the AFQ derivatives directory.
In addition, it calculates the FA from these parameters and stores it in a
different file in the same directory.

:::{note}
The AFQ API computes quantities lazily. This means that DTI parameters
are not computed until they are required. This means that the first
line below is the one that requires time.
:::

The result of the call to `export` is the filename of the corresponding FA
files.

In [ ]:
FA_fname = myafq.export("dti_fa")

We will then use `nibabel` to load the deriviative file and retrieve the
data array.

In [ ]:
FA_img = nib.load(FA_fname)
FA = FA_img.get_fdata()

## Visualize the result with Matplotlib

At this point `FA` is an array, and we can use standard Python tools to
visualize it or perform additional computations with it.

In this case we are going to take an axial slice halfway through the
FA data array and plot using a sequential color map.

:::{note}
The data array is structured as a xyz coordinate system.
:::

In [ ]:
fig, ax = plt.subplots(1)
ax.matshow(FA[:, :, FA.shape[-1] // 2], cmap="viridis")
ax.axis("off")
plt.savefig("FA.png")

In [ ]:
from AFQ.utils.docs import embed_html, embed_image

embed_image("FA.png")

## Recognizing the bundles and calculating tract profiles:

Typically, users of pyAFQ are interested in calculating not only an overall
map of the FA, but also the major white matter pathways (or bundles) and
tract profiles of tissue properties along their length. To trigger the
pyAFQ pipeline that calculates the profiles, users can call the
`export("profiles")` method:

:::{note}
Running the code below triggers the full pipeline of operations
leading to the computation of the tract profiles. Therefore, it
takes a little while to run (about 40 minutes, typically).
:::

In [ ]:
myafq.export("profiles")

## Visualizing the bundles and calculating tract profiles:

The pyAFQ API provides several ways to visualize bundles and profiles.

First, we will run a function that exports an html file that contains
an interactive visualization of the bundles that are segmented.

:::{note}
By default we resample a 100 points within a bundle, however to reduce
processing time we will only resample 50 points.
:::

Once it is done running, it should pop a browser window open and let you
interact with the bundles.

:::{note}
You can hide or show a bundle by clicking the legend, or select a
single bundle by double clicking the legend. The interactive
visualization will also allow you to pan, zoom, and rotate.
:::

In [ ]:
bundle_html = myafq.export("all_bundles_figure")
plotly.io.show(bundle_html[0])

In [ ]:
bundle_html = myafq.export("all_bundles_figure")
embed_html(bundle_html[1])

We can also visualize the tract profiles in all of the bundles. These
plots show both FA (left) and MD (right) laid out anatomically.
To make this plot, it is required that you install with
`pip install pyAFQ[plot]` so that you have the necessary dependencies.

In [ ]:
fig_files = myafq.export("tract_profile_plots")

In [ ]:
embed_image(fig_files[0] + ".png")

## Exporting citations
Finally, we can export the citations for the some of methods used in this
analysis. These are not guaranteed to be comprehensive, but they
should be a good starting point.

In [ ]:
myafq.export("citations")

## References

[^1]: Alexander LM, Escalera J, Ai L, et al. An open resource for
    transdiagnostic research in pediatric mental health and learning
    disorders. Sci Data. 2017;4:170181.

[^2]: Richie-Halford A, Cieslak M, Ai L, et al. An analysis-ready and quality
    controlled resource for pediatric brain white-matter research. Scientific
    Data. 2022;9(1):1-27.

[^3]: Cieslak M, Cook PA, He X, et al. QSIPrep: an integrative platform for
    preprocessing and reconstructing diffusion MRI data. Nat Methods.
    2021;18(7):775-778.

:::{only} html
{download}`Download as Jupyter Notebook <plot_002_participant_afq_api.ipynb>`
:::